# 🤖 Open Manus — Agent Framework

**Open-Source Multi-Agent-System** — Agent-Konfiguration, Tool-Use, Sandbox-Execution und A2A-Protokoll.

## Übersicht

Dieses Notebook demonstriert das Open Manus Agent Framework:
1. **Konfiguration** — LLM-Setup, Agent-Typen, Tool-Auswahl
2. **Agent-Erstellung** — GeneralAgent, CodeAgent, DataAnalysisAgent, BrowserAgent
3. **Tool-Collection** — Web Search, Python Execute, Browser, File Operations
4. **Task-Ausführung** — Tasks an Agenten delegieren und Ergebnisse verfolgen
5. **Sandbox** — Docker-basierte isolierte Code-Ausführung
6. **A2A-Protokoll** — Agent-to-Agent-Kommunikation

> **Repository:** [github.com/mark-baumann/open-manus](https://github.com/mark-baumann/open-manus)

## 1. Umgebung & Konfiguration

In [ ]:
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import Any, Optional

# Projekt-Root zum Pfad hinzufügen
sys.path.insert(0, os.path.abspath("."))

# Core-Imports
from app.config import config as app_config
from app.schema import (
    Message, Memory, Role, ToolCall, ToolChoice,
    AgentState, Function,
)
from app.tool.base import BaseTool, ToolResult
from app.tool.tool_collection import ToolCollection

print("✅ Open Manus Framework geladen!")
print(f"   Workspace: {app_config.workspace_root}")
print(f"   LLM Default Model: {app_config.llm['default']['model']}")
print(f"   LLM API Type: {app_config.llm['default']['api_type']}")

### 1.1 Konfiguration anzeigen & anpassen

In [ ]:
# === Aktuelle Konfiguration ===
print("📋 LLM-Konfiguration:")
for name, settings in app_config.llm.items():
    print(f"\n   [{name}]")
    print(f"   Model:      {settings.model}")
    print(f"   Base URL:   {settings.base_url}")
    print(f"   API Type:   {settings.api_type}")
    print(f"   Max Tokens: {settings.max_tokens}")
    print(f"   Temperature: {settings.temperature}")

print(f"\n📦 Sandbox-Konfiguration:")
if app_config.sandbox:
    sb = app_config.sandbox
    print(f"   Enabled:     {sb.use_sandbox}")
    print(f"   Image:       {sb.image}")
    print(f"   Memory:      {sb.memory_limit}")
    print(f"   Timeout:     {sb.timeout}s")

print(f"\n🔍 Search-Konfiguration:")
if app_config.search_config:
    sc = app_config.search_config
    print(f"   Engine:      {sc.engine}")
    print(f"   Fallbacks:   {sc.fallback_engines}")

print(f"\n🌐 Browser-Konfiguration:")
if app_config.browser_config:
    bc = app_config.browser_config
    print(f"   Headless:    {bc.headless}")
    print(f"   Max Content: {bc.max_content_length}")

## 2. Schema & Datenmodell

Open Manus verwendet Pydantic-Modelle für Messages, Memory und Agent-State.

In [ ]:
# === Message-System ===
print("📝 Message-System:")

# User Message
user_msg = Message.user_message("Erstelle eine Python-Funktion für Fibonacci-Zahlen.")
print(f"   User:     {user_msg.role.value} → {user_msg.content[:60]}...")

# System Message
system_msg = Message.system_message("Du bist ein hilfreicher Python-Experte.")
print(f"   System:   {system_msg.role.value} → {system_msg.content[:60]}...")

# Assistant Message mit Tool Call
tool_call = ToolCall(
    id="call_1",
    type="function",
    function=Function(
        name="python_execute",
        arguments='{"code": "def fib(n): return n if n <= 1 else fib(n-1) + fib(n-2)"}',
    ),
)
assistant_msg = Message.assistant_message(content="Ich führe den Code aus.")
assistant_msg.tool_calls = [tool_call]
print(f"   Assistant: {assistant_msg.role.value} → {len(assistant_msg.tool_calls)} tool call(s)")

# Tool Message
tool_msg = Message.tool_message(
    content="Code erfolgreich ausgeführt. Ergebnis: fib(10) = 55",
    name="python_execute",
    tool_call_id="call_1",
)
print(f"   Tool:      {tool_msg.role.value} → {tool_msg.content[:60]}...")

# Message zu Dict
print(f"\n📋 Message.to_dict():\n{json.dumps(user_msg.to_dict(), indent=2, ensure_ascii=False)}")

In [ ]:
# === Memory-System ===
memory = Memory(max_messages=100)

# Nachrichten hinzufügen
memory.add_message(system_msg)
memory.add_message(user_msg)
memory.add_message(assistant_msg)
memory.add_message(tool_msg)

print(f"🧠 Memory: {len(memory.messages)} Nachrichten")
print(f"   Max: {memory.max_messages}")

# Letzte N Nachrichten
recent = memory.get_recent_messages(2)
print(f"\n📋 Letzte 2 Nachrichten:")
for msg in recent:
    print(f"   [{msg.role.value}] {str(msg.content)[:80]}...")

# Memory als Dict-Liste (für API-Calls)
dict_list = memory.to_dict_list()
print(f"\n📋 to_dict_list(): {len(dict_list)} Einträge")

In [ ]:
# === AgentState ===
print("🔄 Agent-Zustände:")
for state in AgentState:
    print(f"   {state.name}: {state.value}")

# === ToolChoice ===
print("\n🔧 ToolChoice-Optionen:")
for choice in ToolChoice:
    print(f"   {choice.name}: {choice.value}")

## 3. Tool-System

Open Manus bietet eine umfangreiche Tool-Collection mit Web Search, Python Execute, Browser und File Operations.

In [ ]:
# === Tool-Basisklasse ===
print("🔧 Tool-Basisklasse (BaseTool):")
print(f"   Abstrakte Basis: {BaseTool.__bases__}")
print(f"   Methoden: execute() [abstract], to_param(), success_response(), fail_response()")

# === ToolResult ===
print("\n📊 ToolResult:")
success = ToolResult(output="Operation erfolgreich!")
failure = ToolResult(error="Datei nicht gefunden.")
print(f"   Success: output='{success.output}', error={success.error}")
print(f"   Failure: output={failure.output}, error='{failure.error}'")
print(f"   Bool-Check: success={bool(success)}, failure={bool(failure)}")

# === ToolCollection ===
print("\n📦 ToolCollection:")
print(f"   Verwaltet mehrere Tools und bietet to_params() für API-Calls.")

### 3.1 Eigenes Tool erstellen

Demonstriere, wie man ein benutzerdefiniertes Tool erstellt:

In [ ]:
from pydantic import Field


class CalculatorTool(BaseTool):
    """Ein einfacher Rechner-Tool."""

    name: str = "calculator"
    description: str = "Führt mathematische Berechnungen durch. Unterstützt +, -, *, /, **."
    parameters: dict = Field(default_factory=lambda: {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "Der mathematische Ausdruck (z.B. '2 + 3 * 4')",
            },
        },
        "required": ["expression"],
    })

    async def execute(self, expression: str) -> ToolResult:
        """Führt die Berechnung aus."""
        try:
            # Sicherer Eval mit eingeschränktem Namespace
            result = eval(expression, {"__builtins__": {}}, {})
            return self.success_response({"expression": expression, "result": result})
        except Exception as e:
            return self.fail_response(f"Berechnungsfehler: {e}")


# === Tool testen ===
calc = CalculatorTool()
print(f"🔧 Tool: {calc.name}")
print(f"   Beschreibung: {calc.description[:80]}...")
print(f"\n📋 to_param():\n{json.dumps(calc.to_param(), indent=2, ensure_ascii=False)}")

# Tool ausführen
result = asyncio.run(calc.execute(expression="2 + 3 * 4"))
print(f"\n🚀 Ausführung: 2 + 3 * 4")
print(f"   Ergebnis: {result.output}")

### 3.2 Verfügbare Tools

Übersicht der im Framework enthaltenen Tools:

In [ ]:
# === Tool-Übersicht ===
tools_overview = [
    {"Name": "web_search", "Kategorie": "Search", "Beschreibung": "Websuche via Google, Bing, DuckDuckGo, Baidu"},
    {"Name": "python_execute", "Kategorie": "Code", "Beschreibung": "Python-Code in Sandbox ausführen"},
    {"Name": "browser", "Kategorie": "Browser", "Beschreibung": "Headless-Browser für Web-Interaktion"},
    {"Name": "str_replace_editor", "Kategorie": "File", "Beschreibung": "Text-Editor mit Suchen & Ersetzen"},
    {"Name": "terminate", "Kategorie": "Control", "Beschreibung": "Agent-Ausführung beenden"},
    {"Name": "chart_visualization", "Kategorie": "Viz", "Beschreibung": "Diagramme und Visualisierungen erstellen"},
    {"Name": "sandbox_terminal", "Kategorie": "Sandbox", "Beschreibung": "Terminal-Befehle in Docker-Sandbox"},
    {"Name": "sandbox_browser", "Kategorie": "Sandbox", "Beschreibung": "Browser in isolierter Sandbox"},
]

print("🛠️  Tool-Collection:\n")
print(f"{'Name':<25s} {'Kategorie':<12s} Beschreibung")
print("-" * 80)
for tool in tools_overview:
    print(f"{tool['Name']:<25s} {tool['Kategorie']:<12s} {tool['Beschreibung']}")

## 4. Agent-Konfiguration & Task-Ausführung

Open Manus bietet vier spezialisierte Agent-Typen.

In [ ]:
# === Agent-Typen ===
agent_types = [
    {
        "name": "GeneralAgent",
        "description": "Allzweck-Agent für allgemeine Aufgaben",
        "tools": ["web_search", "python_execute", "browser", "str_replace_editor"],
        "use_cases": ["Recherche", "Textanalyse", "Allgemeine Fragen"],
    },
    {
        "name": "CodeAgent",
        "description": "Spezialisiert auf Code-Generierung und -Ausführung",
        "tools": ["python_execute", "str_replace_editor", "sandbox_terminal"],
        "use_cases": ["Code schreiben", "Debugging", "Refactoring"],
    },
    {
        "name": "DataAnalysisAgent",
        "description": "Datenanalyse und Visualisierung",
        "tools": ["python_execute", "chart_visualization", "web_search"],
        "use_cases": ["Datenanalyse", "Visualisierung", "Statistik"],
    },
    {
        "name": "BrowserAgent",
        "description": "Web-Interaktion und Scraping",
        "tools": ["browser", "web_search", "sandbox_browser"],
        "use_cases": ["Web-Scraping", "Formulare ausfüllen", "Screenshots"],
    },
]

print("🤖 Agent-Typen:\n")
for agent in agent_types:
    print(f"{'─' * 60}")
    print(f"  {agent['name']}")
    print(f"  {agent['description']}")
    print(f"  Tools: {', '.join(agent['tools'])}")
    print(f"  Use Cases: {', '.join(agent['use_cases'])}")

### 4.1 Agent ausführen (Manus)

Der Haupt-Agent `Manus` kombiniert alle Fähigkeiten:

In [ ]:
# === Manus-Agent ausführen ===
print("🚀 Manus-Agent Demo:\n")

try:
    from app.agent.manus import Manus

    async def run_manus_demo():
        agent = await Manus.create()
        try:
            prompt = "Was ist die Hauptstadt von Frankreich? Gib eine kurze Antwort."
            print(f"   Prompt: {prompt}")
            await agent.run(prompt)
            print("\n✅ Task abgeschlossen!")
        finally:
            await agent.cleanup()

    # In Jupyter: asyncio.run() oder await in async-Cell
    # asyncio.run(run_manus_demo())
    print("   (Agent-Ausführung benötigt API-Key — in Jupyter mit await ausführen)")

except ImportError as e:
    print(f"   ⚠️  Manus-Agent nicht verfügbar: {e}")
except Exception as e:
    print(f"   ⚠️  Fehler: {e}")

### 4.2 Task-Delegation (Flow)

Tasks können über den Flow-Mechanismus an Agenten delegiert werden:

In [ ]:
# === Flow-Konfiguration ===
print("🔄 Run-Flow-Konfiguration:")
if app_config.run_flow_config:
    rfc = app_config.run_flow_config
    print(f"   Data Analysis Agent: {rfc.use_data_analysis_agent}")

print("\n📋 Flow-Beispiel:")
print("""
   Task: "Analysiere die Verkaufsdaten und erstelle einen Report"
   
   1. GeneralAgent → Zerlegt Task in Subtasks
   2. DataAnalysisAgent → Analysiert CSV-Daten
   3. CodeAgent → Schreibt Analyse-Script
   4. GeneralAgent → Erstellt finalen Report
""")

## 5. Sandbox-Execution

Docker-basierte isolierte Ausführungsumgebung für sicheres Code-Running.

In [ ]:
# === Sandbox-Konfiguration ===
print("📦 Sandbox-Execution:\n")

sandbox_config = {
    "image": "python:3.12-slim",
    "work_dir": "/workspace",
    "memory_limit": "512m",
    "cpu_limit": 1.0,
    "timeout": 300,
    "network_enabled": False,
}

print("   Konfiguration:")
for key, val in sandbox_config.items():
    print(f"   {key:<20s}: {val}")

print("\n   Sicherheits-Features:")
print("   • Isolierter Container (kein Host-Zugriff)")
print("   • Netzwerk deaktiviert (optional)")
print("   • Memory/CPU-Limits")
print("   • Timeout pro Befehl")
print("   • Read-only Root-Filesystem (optional)")

print("\n   Verwendung:")
print("   from app.sandbox import SandboxManager")
print("   sandbox = SandboxManager(config)")
print("   result = await sandbox.execute('python script.py')")

## 6. A2A-Protokoll (Agent-to-Agent)

Das A2A-Protokoll ermöglicht die Kommunikation zwischen mehreren Agenten.

In [ ]:
# === A2A-Protokoll ===
print("🔗 Agent-to-Agent (A2A) Protokoll:\n")

print("   Architektur:")
print("   ┌──────────┐    A2A-Protokoll    ┌──────────┐")
print("   │ Agent A  │ ◄──────────────────► │ Agent B  │")
print("   │ (Master) │    Task-Delegation   │ (Worker) │")
print("   └──────────┘                      └──────────┘")

print("\n   Komponenten:")
print("   • protocol/a2a/app/agent.py        — A2A-Agent-Implementierung")
print("   • protocol/a2a/app/agent_executor.py — Task-Ausführung")
print("   • protocol/a2a/app/main.py          — A2A-Server")

print("\n   Verwendung:")
print("   python run_mcp.py          # MCP-Integration starten")
print("   python run_mcp_server.py   # MCP-Server starten")
print("   python run_flow.py         # Flow mit A2A ausführen")

## 7. Handbook-Compliance

Regelbasierte Validierung von Agent-Ausgaben gegen ein Compliance-Handbook.

In [ ]:
# === Handbook-Compliance Demo ===
print("📋 Handbook-Compliance:\n")

try:
    from app.handbook_compliance import HandbookCompliance

    print("   Modul: app/handbook_compliance.py")
    print("   Funktion: Validiert Agent-Ausgaben gegen definierte Regeln")
    print()
    print("   Beispiel-Regeln:")
    print("   • Keine persönlichen Daten (DSGVO)")
    print("   • Keine schädlichen Anweisungen")
    print("   • Korrektes Ausgabeformat")
    print("   • Einhaltung von Unternehmensrichtlinien")

except ImportError:
    print("   ⚠️  Handbook-Compliance-Modul nicht geladen.")
    print("   Verfügbar unter: app/handbook_compliance.py")

## 8. MCP-Integration

Model Context Protocol für externe Tool-Integration.

In [ ]:
# === MCP-Konfiguration ===
print("🔌 MCP (Model Context Protocol):\n")

if app_config.mcp_config:
    mcp = app_config.mcp_config
    print(f"   Server-Referenz: {mcp.server_reference}")
    print(f"   Server: {len(mcp.servers)} konfiguriert")
    for name, server in mcp.servers.items():
        print(f"   • {name}: type={server.type}")
        if server.url:
            print(f"     URL: {server.url}")
        if server.command:
            print(f"     Command: {server.command} {' '.join(server.args)}")

print("\n   MCP-Server starten:")
print("   python run_mcp.py          # MCP-Client")
print("   python run_mcp_server.py   # MCP-Server")

## 9. Zusammenfassung

### Open Manus Architektur

```
┌─────────────────────────────────────────────────────────────┐
│                     OPEN MANUS                              │
├─────────────────────────────────────────────────────────────┤
│  AGENTEN                    TOOLS                           │
│  ┌──────────────┐          ┌──────────────────────────┐    │
│  │ GeneralAgent │          │ web_search               │    │
│  │ CodeAgent    │──────────│ python_execute           │    │
│  │ DataAnalysis │          │ browser                  │    │
│  │ BrowserAgent │          │ str_replace_editor       │    │
│  └──────────────┘          │ chart_visualization      │    │
│                            │ terminate                 │    │
│  INFRASTRUKTUR             └──────────────────────────┘    │
│  ┌──────────────┐                                          │
│  │ Sandbox      │  Docker-Container für sicheres Coding    │
│  │ A2A-Protocol │  Agent-zu-Agent-Kommunikation            │
│  │ MCP          │  Model Context Protocol                  │
│  │ Handbook     │  Compliance-Regelwerk                    │
│  └──────────────┘                                          │
└─────────────────────────────────────────────────────────────┘
```

### CLI-Aufruf

```bash
# Streamlit-App
streamlit run app/app.py

# Haupt-CLI
python main.py --prompt "Erstelle eine Datenanalyse"

# Flow ausführen
python run_flow.py

# MCP-Server
python run_mcp.py
python run_mcp_server.py

# Sandbox
python sandbox_main.py
```

### Konfigurationsdateien

| Datei | Zweck |
|---|---|
| `config/config.toml` | LLM, Sandbox, Browser, Search |
| `config/mcp.json` | MCP-Server-Konfiguration |
| `app/config.py` | Pydantic-Modelle + Singleton |
| `app/schema.py` | Message, Memory, AgentState |

> **Repository:** [github.com/mark-baumann/open-manus](https://github.com/mark-baumann/open-manus)